In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz
from ipywidgets import Dropdown, FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# POLE-ZERO PLACEMENT METHOD
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.pz-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.pz-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.pz-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.pz-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.pz-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
}

.pz-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.pz-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:13px !important;
}

.jupyter-widgets input,
.jupyter-widgets select{
    font-size:12.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="pz-root">

<div class="pz-header">
Pole-Zero Placement and Frequency Response
</div>

<div class="pz-doc">

This notebook illustrates the basic idea of the <b>pole-zero placement method</b>.
A zero placed on or near the unit circle reduces the magnitude response at the
corresponding frequency, while a pole placed close to the unit circle increases it.

For low-pass and high-pass filters a real pole <b>α</b> is used.
For band-pass and band-stop filters the conjugate poles are

<b>p₁,₂ = r e<sup>±jθ</sup></b>.

The parameter <b>r</b> controls how close the poles are to the unit circle and therefore
the sharpness of the response, while <b>θ</b> determines the resonance or notch frequency.
Move the controls and observe simultaneously the pole-zero geometry and the resulting
magnitude response.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

filter_type = Dropdown(options=['Low-pass','High-pass','Band-pass','Band-stop'],value='Low-pass',description='Type:',style={'description_width':'35px'},layout=Layout(width='175px'))

alpha_slider = FloatSlider(value=0.90,min=-0.95,max=0.95,step=0.01,description='α:',continuous_update=True,readout_format='.2f',style={'description_width':'20px'},layout=Layout(width='195px'))

r_slider = FloatSlider(value=0.92,min=0.80,max=0.99,step=0.01,description='r:',continuous_update=True,readout_format='.2f',style={'description_width':'20px'},layout=Layout(width='190px'))

theta_slider = FloatSlider(value=0.30,min=0.05,max=0.95,step=0.01,description='θ/π:',continuous_update=True,readout_format='.2f',style={'description_width':'35px'},layout=Layout(width='215px'))

control_title = HTML('<div class="pz-title" style="margin:0;">Parameters</div>',layout=Layout(width='90px'))

controls = HBox([control_title,filter_type,alpha_slider,r_slider,theta_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='8px 10px',margin='0 0 7px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 7px 0'))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9.0,4.0))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. POLE-ZERO PLOT
# ============================================================

theta_circle = np.linspace(0,2*np.pi,1000)

ax1.axhline(0,color='black',linewidth=0.8)
ax1.axvline(0,color='black',linewidth=0.8)
ax1.plot(np.cos(theta_circle),np.sin(theta_circle),'--',linewidth=1.1,label='Unit circle')

pole_plot, = ax1.plot([],[],'rx',markersize=8,markeredgewidth=1.8,label='Poles')
zero_plot, = ax1.plot([],[],'bo',markersize=7,markerfacecolor='none',markeredgewidth=1.6,label='Zeros')

ax1.set_xlim(-1.2,1.2)
ax1.set_ylim(-1.2,1.2)
ax1.set_aspect('equal',adjustable='box')

ax1.set_title('Pole-Zero Placement')
ax1.set_xlabel(r'$\Re\{z\}$')
ax1.set_ylabel(r'$\Im\{z\}$')

ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# 2. MAGNITUDE RESPONSE
# ============================================================

magnitude_line, = ax2.plot([],[],color='red',linewidth=1.5,label='Magnitude response')

frequency_marker = ax2.axvline(0.30,linestyle='--',linewidth=1.0,label=r'$\theta/\pi$')

ax2.set_xlim(0,1)
ax2.set_ylim(0,1.40)

ax2.set_title('Magnitude Response')
ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax2.set_ylabel(r'$|H(e^{j\omega})|$')

ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.91,bottom=0.22,wspace=0.28)

# ============================================================
# UPDATE
# ============================================================

def update_filter(change=None):

    kind = filter_type.value
    alpha = alpha_slider.value
    r = r_slider.value
    theta = theta_slider.value*np.pi

    # --------------------------------------------------------
    # LOW-PASS FILTER
    # --------------------------------------------------------

    if kind == 'Low-pass':

        alpha_slider.disabled = False
        r_slider.disabled = True
        theta_slider.disabled = True

        poles = np.array([alpha+0j])
        zeros = np.array([-1.0+0j])

        H0 = (1-alpha)/2

        b = np.array([H0,H0])
        a = np.array([1.0,-alpha])

        frequency_marker.set_visible(False)

        description = 'The zero at z = -1 suppresses the response at ω = π. Moving the pole toward z = +1 increases the low-frequency selectivity.'

    # --------------------------------------------------------
    # HIGH-PASS FILTER
    # --------------------------------------------------------

    elif kind == 'High-pass':

        alpha_slider.disabled = False
        r_slider.disabled = True
        theta_slider.disabled = True

        poles = np.array([alpha+0j])
        zeros = np.array([1.0+0j])

        H0 = (1+alpha)/2

        b = np.array([H0,-H0])
        a = np.array([1.0,-alpha])

        frequency_marker.set_visible(False)

        description = 'The zero at z = +1 suppresses the response at ω = 0. Moving the pole toward z = -1 increases the high-frequency selectivity.'

    # --------------------------------------------------------
    # BAND-PASS FILTER
    # --------------------------------------------------------

    elif kind == 'Band-pass':

        alpha_slider.disabled = True
        r_slider.disabled = False
        theta_slider.disabled = False

        poles = np.array([r*np.exp(1j*theta),r*np.exp(-1j*theta)])
        zeros = np.array([1.0+0j,-1.0+0j])

        H0 = (1-r)*np.sqrt(1-2*r*np.cos(2*theta)+r**2)/(2*np.abs(np.sin(theta)))

        b = H0*np.array([1.0,0.0,-1.0])
        a = np.array([1.0,-2*r*np.cos(theta),r**2])

        frequency_marker.set_xdata([theta/np.pi,theta/np.pi])
        frequency_marker.set_visible(True)

        description = 'Zeros at z = ±1 suppress DC and the Nyquist frequency. The conjugate poles create a resonance near ω = θ; increasing r makes the resonance sharper.'

    # --------------------------------------------------------
    # BAND-STOP FILTER
    # --------------------------------------------------------

    else:

        alpha_slider.disabled = True
        r_slider.disabled = False
        theta_slider.disabled = False

        poles = np.array([r*np.exp(1j*theta),r*np.exp(-1j*theta)])
        zeros = np.array([np.exp(1j*theta),np.exp(-1j*theta)])

        H0 = (1-2*r*np.cos(theta)+r**2)/(2*(1-np.cos(theta)))

        b = H0*np.array([1.0,-2*np.cos(theta),1.0])
        a = np.array([1.0,-2*r*np.cos(theta),r**2])

        frequency_marker.set_xdata([theta/np.pi,theta/np.pi])
        frequency_marker.set_visible(True)

        description = 'The conjugate zeros on the unit circle force the response to zero at ω = θ. Nearby poles control the width and sharpness of the notch.'

    # --------------------------------------------------------
    # FREQUENCY RESPONSE
    # --------------------------------------------------------

    omega,H = freqz(b,a,worN=32768)

    magnitude = np.abs(H)

    # --------------------------------------------------------
    # UPDATE PLOTS
    # --------------------------------------------------------

    pole_plot.set_data(np.real(poles),np.imag(poles))
    zero_plot.set_data(np.real(zeros),np.imag(zeros))

    magnitude_line.set_data(omega/np.pi,magnitude)

    # --------------------------------------------------------
    # COMPACT NUMERICAL INFORMATION
    # --------------------------------------------------------

    if kind in ['Low-pass','High-pass']:

        parameter_text = f'α = <b>{alpha:.3f}</b> &nbsp;&nbsp; H₀ = <b>{H0:.6f}</b>'

    else:

        parameter_text = f'r = <b>{r:.3f}</b> &nbsp;&nbsp; θ = <b>{theta/np.pi:.3f}π</b> &nbsp;&nbsp; H₀ = <b>{H0:.6f}</b>'

    b_text = ', '.join([f'{value:.6f}' for value in b])
    a_text = ', '.join([f'{value:.6f}' for value in a])

    info.value = f"""
    <div class="pz-root">

    <div class="pz-box">

    <div class="pz-cols">

    <div class="pz-col">
    <b>{kind} filter</b><br>
    {parameter_text}
    </div>

    <div class="pz-col">
    <b>Transfer-function coefficients</b><br>
    b = [{b_text}]<br>
    a = [{a_text}]
    </div>

    </div>

    <div style="margin-top:6px;">
    {description}
    </div>

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

filter_type.observe(update_filter,names='value')
alpha_slider.observe(update_filter,names='value')
r_slider.observe(update_filter,names='value')
theta_slider.observe(update_filter,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_filter()